# Mental Health AI Screening System - Hybrid Model Training

This notebook demonstrates the training pipeline for the hybrid CNN-LSTM-Random Forest model for mental health screening as described in the research. The hybrid model has the ability to outshine separate algorithms and achieves a reported 99.3% accuracy.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LSTM, Conv1D, MaxPooling1D, Flatten, Input, concatenate, Dropout
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("TensorFlow Version:", tf.__version__)
print("Libraries loaded successfully.")

## 2. Data Preparation (Mock Data)
We generate mock multimodal data to represent visual features (CNN), sequential text features (LSTM), and structured questionnaire data (Random Forest).

In [ ]:
# Generate mock data for 1000 samples
num_samples = 1000

# 1. Visual/Physiological data for CNN (e.g., 100 timesteps, 3 channels)
X_cnn = np.random.rand(num_samples, 100, 3)

# 2. Text sequence data for LSTM (e.g., 50 words, 100 embedding dim)
X_lstm = np.random.rand(num_samples, 50, 100)

# 3. Structured questionnaire data for Random Forest (e.g., 10 questions)
X_rf = np.random.randint(0, 4, size=(num_samples, 10))

# Labels: 0 (Low Risk), 1 (Moderate Risk), 2 (High Risk)
y = np.random.randint(0, 3, size=(num_samples, 1))

# Split the data
split_idx = int(0.8 * num_samples)
X_cnn_train, X_cnn_test = X_cnn[:split_idx], X_cnn[split_idx:]
X_lstm_train, X_lstm_test = X_lstm[:split_idx], X_lstm[split_idx:]
X_rf_train, X_rf_test = X_rf[:split_idx], X_rf[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("Data prepared and split.")

## 3. Build the Neural Network Feature Extractors (CNN & LSTM)

In [ ]:
# CNN Branch
input_cnn = Input(shape=(100, 3), name="cnn_input")
x1 = Conv1D(filters=64, kernel_size=3, activation='relu')(input_cnn)
x1 = MaxPooling1D(pool_size=2)(x1)
x1 = Flatten()(x1)
x1 = Dense(32, activation='relu')(x1)
cnn_model = Model(inputs=input_cnn, outputs=x1, name="CNN_Extractor")

# LSTM Branch
input_lstm = Input(shape=(50, 100), name="lstm_input")
x2 = LSTM(64, return_sequences=False)(input_lstm)
x2 = Dense(32, activation='relu')(x2)
lstm_model = Model(inputs=input_lstm, outputs=x2, name="LSTM_Extractor")

# Combine CNN and LSTM for joint representation
combined_features = concatenate([cnn_model.output, lstm_model.output])
z = Dense(64, activation='relu')(combined_features)
z = Dropout(0.3)(z)
z = Dense(3, activation='softmax', name="nn_output")(z)

hybrid_nn = Model(inputs=[input_cnn, input_lstm], outputs=z)
hybrid_nn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hybrid_nn.summary()

## 4. Train the Neural Network

In [ ]:
# Train CNN-LSTM model
history = hybrid_nn.fit(
    [X_cnn_train, X_lstm_train],
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=([X_cnn_test, X_lstm_test], y_test),
    verbose=1
)

## 5. Extract Features for Random Forest
We use the trained Neural Networks to extract intermediate features and combine them with the structured questionnaire data. Then we feed it into the Random Forest classifier for the final prediction.

In [ ]:
# Create an extractor model that outputs the combined dense layer
feature_extractor = Model(inputs=hybrid_nn.input, outputs=hybrid_nn.layers[-2].output)

# Extract features
nn_features_train = feature_extractor.predict([X_cnn_train, X_lstm_train])
nn_features_test = feature_extractor.predict([X_cnn_test, X_lstm_test])

# Combine with structured RF data
final_X_train = np.hstack((nn_features_train, X_rf_train))
final_X_test = np.hstack((nn_features_test, X_rf_test))

print("Final training data shape for Random Forest:", final_X_train.shape)

## 6. Train Random Forest Meta-Classifier
The Random Forest provides the final interpretable, stable output.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(final_X_train, y_train.ravel())

y_pred = rf_model.predict(final_X_test)

print("Hybrid Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## Conclusion
The hybrid CNN-LSTM-Random Forest framework successfully synergizes the strengths of CNN's feature extraction, LSTM's sequential pattern learning, and Random Forest's robust classification. This approach is aligned with the latest trends in multimodal mental health screening, aiming to provide an inclusive and highly accurate automated system.